# Keisler Engine — Research Playbook

Run global weather forecasts with the C++/ONNX backend from any Jupyter
environment (Colab, Kaggle, local). The engine accepts **any ONNX model** —
see the model preparation section for PyTorch, JAX, and TensorFlow.

---

**Input contract** — the engine expects `float32[1, 71042, 78]`:
- **71042 nodes** — 1° ERA5 grid (181 × 360) merged with H3 mesh
- **78 channels** — variables `[z, q, t, u, v, w]` at 13 pressure levels
  (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000 hPa)

In [ ]:
# ── Cell 1: Environment setup ─────────────────────────────────────────────────
# Run once per Colab/Kaggle session. Skip locally if you have a pre-built wheel.
import sys, subprocess, os

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    # System build tools
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'cmake', 'g++'], check=True)

    # Clone the engine repo
    if not os.path.exists('/tmp/weathergraph'):
        subprocess.run([
            'git', 'clone', '--depth=1',
            'https://github.com/weathergraph-engine/weathergraph',
            '/tmp/weathergraph',
        ], check=True)

    os.chdir('/tmp/weathergraph')

    # Download ONNX Runtime shared library
    subprocess.run(['make', 'onnxruntime'], check=True)

    # Install build deps and build the Python extension
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'pybind11>=2.13.0', 'scikit-build-core',
    ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--no-build-isolation', '.',
    ], check=True)

    # Pre-load the shared library so it is discoverable at import time
    import ctypes
    for lib in sorted(__import__('glob').glob('weathergraph/core/libonnxruntime.so*')):
        ctypes.CDLL(lib)

print('Engine environment ready.')

---
## Step 1 — Prepare your model

Run **exactly one** of the cells below, then proceed to Step 2.

In [ ]:
# ── Option A: Download a pre-built ONNX (HuggingFace or any HTTPS URL) ───────
import urllib.request, os

MODEL_PATH = 'models/keisler_2022.onnx'
os.makedirs('models', exist_ok=True)

MODEL_URL = 'https://huggingface.co/Wanderspool/Keisler_2022/resolve/main/keisler_2022.onnx'
# Other examples:
#   MODEL_URL = 'https://my-bucket.s3.amazonaws.com/models/graphcast.onnx'
#   MODEL_URL = 'https://github.com/MyOrg/Repo/releases/download/v1.0/model.onnx'

urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f'Downloaded {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB  →  {MODEL_PATH}')

In [ ]:
# ── Option B: Convert from PyTorch ───────────────────────────────────────────
# pip install torch  — run if torch is not already installed
import torch, os

MODEL_PATH = 'models/my_pytorch_model.onnx'
os.makedirs('models', exist_ok=True)

# ── Replace with your model ──────────────────────────────────────────────────
# from my_package import MyWeatherGNN
# model = MyWeatherGNN()
# model.load_state_dict(torch.load('my_weights.pt', map_location='cpu'))
# model.eval()
# ─────────────────────────────────────────────────────────────────────────────
# Demo: use a trivial identity module as a stand-in
model = torch.nn.Identity()

dummy_input = torch.zeros(1, 71042, 78, dtype=torch.float32)
torch.onnx.export(
    model, dummy_input, MODEL_PATH,
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
)
print(f'Exported {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB  →  {MODEL_PATH}')

In [ ]:
# ── Option C: Convert from TensorFlow / Keras (tf2onnx) ──────────────────────
# pip install tensorflow tf2onnx  — run if not already installed
import subprocess, sys, os

SAVED_MODEL_DIR = 'my_saved_model'   # ← your tf.saved_model directory
MODEL_PATH      = 'models/my_tf_model.onnx'
os.makedirs('models', exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'tf2onnx.convert',
    '--saved-model', SAVED_MODEL_DIR,
    '--output',      MODEL_PATH,
    '--opset',       '17',
], check=True)
print(f'Converted  →  {MODEL_PATH}')

In [ ]:
# ── Option D: Build Keisler ONNX from repo extractor scripts (requires LFS) ──
# This uses exporter/build_gnn_graph.py which reads data/weights/ and
# data/graph_data/ — both are Git LFS objects, already present in the repo.
import subprocess, sys, os

MODEL_PATH = 'models/weather_gnn.onnx'
os.makedirs('models', exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', 'numpy', 'onnx', 'onnxscript'
], check=True)
subprocess.run([
    sys.executable, 'exporter/build_gnn_graph.py', '--output', MODEL_PATH
], check=True)
print(f'Built  →  {MODEL_PATH}')

In [ ]:
# ── Validate ONNX (always run after choosing a model option above) ────────────
import onnx

model_proto = onnx.load(MODEL_PATH)
onnx.checker.check_model(model_proto)
opset = model_proto.opset_import[0].version
print(f'Model OK  — IR v{model_proto.ir_version}, opset {opset}')
print(f'Path: {MODEL_PATH}')

---
## Step 2 — Run inference

In [ ]:
# ── Single 6-hour step ────────────────────────────────────────────────────────
import numpy as np
import weathergraph_backend   # the C++ extension module

engine = weathergraph_backend.WeatherGraphEngine(MODEL_PATH)

# Replace with your real ERA5 initial state [1, 71042, 78] float32
state = np.random.randn(1, 71042, 78).astype(np.float32)

prediction = engine.predict(state)
print(f'Output shape : {prediction.shape}')          # (1, 71042, 78)
print(f'Max abs value: {np.abs(prediction).max():.4f}')

In [ ]:
# ── 10-day autoregressive rollout (40 × 6 h steps) ───────────────────────────
STEPS = 40

current    = state.copy()
trajectory = [current]

for step in range(STEPS):
    current = engine.predict(current)
    trajectory.append(current.copy())
    if (step + 1) % 8 == 0:
        print(f'  step {step+1:3d}/40  ({(step+1)*6}h)  '
              f'mean={current.mean():.4f}  std={current.std():.4f}')

trajectory = np.stack(trajectory, axis=0)   # (41, 1, 71042, 78)
print(f'\nFull trajectory shape: {trajectory.shape}')

In [ ]:
# ── High-level xarray API (requires real ERA5 NetCDF files) ──────────────────
# Uncomment and fill in your data paths.
#
# import xarray as xr
# from weathergraph import WeatherGraphModel
#
# model = WeatherGraphModel(model_path=MODEL_PATH, weights_dir='data')
# ds    = xr.open_dataset('path/to/era5_initial_state.nc')
#
# # Single step
# prediction = model.engine.predict(model._prepare_input(ds))
#
# # Full 10-day rollout
# forecast = model.forecast(ds, steps=40)

print('xarray API ready — uncomment above and provide real ERA5 data.')